# 1 - EXPLORAÇÃO INICIAL

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import learning_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
url = 'https://raw.githubusercontent.com/alvaroriz/datascience_datasets/refs/heads/main/spotfy_churn.csv'
base = pd.read_csv(url)
base.head()

In [ ]:
base.shape

In [ ]:
base.columns

In [ ]:
# Verificando o tipo de cada variável
base.dtypes

In [ ]:
# Verificar se há duplicatas
base.duplicated().sum()

In [ ]:
# Verificar valores ausentes
base.isnull().sum()

In [ ]:
# Outliers
lista = []
for i in base.columns:
  if base.dtypes[i] == 'int64' or base.dtypes[i] == 'float64':
    if i == 'user_id' or i == 'is_churned':
      pass
    else:
      lista.append(i)
# Configurar o layout dos subplots
fig, axes = plt.subplots(nrows=len(lista)//3 + 1, ncols=3, figsize=(15, 10))
axes = axes.flatten()  # Achatar o array de eixos para facilitar a iteração

# Criar um boxplot para cada variável
for idx, coluna in enumerate(lista):
    if idx < len(axes):
        sns.boxplot(y=base[coluna], ax=axes[idx])
        axes[idx].set_title(f'Boxplot de {coluna}')
        axes[idx].set_xlabel('')

# Remover eixos vazios se houver
for idx in range(len(lista), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

In [ ]:
# offline listening é uma variável binária
base['offline_listening'].value_counts()

In [ ]:
# Estatísticas descritivas para identificar outliers
for coluna in lista:
    Q1 = base[coluna].quantile(0.25)
    Q3 = base[coluna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = base[(base[coluna] < limite_inferior) | (base[coluna] > limite_superior)]
    print(f"{coluna}: {len(outliers)} outliers")

In [ ]:
# Investigando melhor a variável ads_listened_per_week
# violin plot
sns.violinplot(x=base['ads_listened_per_week'])
plt.show()

In [ ]:
base['ads_listened_per_week'].describe()

In [ ]:
(base['ads_listened_per_week'] > 12.5).value_counts()

In [ ]:
investigacao = base[base['ads_listened_per_week'] >= 12.5]
investigacao['subscription_type'].value_counts()

In [ ]:
# correlação entre ads_listened_per_week e listening_time
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=base['ads_listened_per_week'],
    y=base['listening_time'],
    hue=base['subscription_type'],
    palette='viridis',
    alpha=0.7
)
plt.title('Tempo de Escuta vs Anúncios por Semana (por plano de assinatura)')
plt.xlabel('Anúncios por Semana')
plt.ylabel('Tempo de Escuta (minutos)')
plt.legend(title='Gênero', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# matriz de correlação
numeric_investigacao = investigacao.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_investigacao.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlação')
plt.show()

# 2 - PRÉ-PROCESSAMENTO

In [ ]:
# Dividindo em base de trabalho e base de validaçao
validacao = base.sample(frac=0.2, random_state=42)
base = base.drop(validacao.index)
print(f'Base de trabalho: {base.shape}')
print(f'Base de validação: {validacao.shape}')

In [ ]:
# Divisão da base de trabalho em treino e teste
base_train, base_test = train_test_split(base, test_size=0.2, random_state=42)
X_train = base_train.drop('is_churned', axis=1)
y_train = base_train['is_churned']

In [ ]:
base_train.shape, base_test.shape

In [ ]:
X_train.head()

In [ ]:
# verificar cardinalidade das variáveis categóricas
for coluna in X_train.columns:
  if X_train.dtypes[coluna] == 'object':
    print(f'{coluna}: {X_train[coluna].nunique()}')

In [ ]:
X_train['subscription_type'].value_counts()

In [ ]:
# Ordinal enconding em subscription_type
subscription_mapping = {'Free': 0, 'Student': 1, 'Family': 2,'Premium':3}
X_train['subscription_type'] = X_train['subscription_type'].map(subscription_mapping)
X_train['subscription_type'].value_counts()

In [ ]:
# One hot enconding com k -1 para gender e device_type
X_train = pd.get_dummies(X_train, columns=['gender', 'device_type'],drop_first=True)
X_train.head()

In [ ]:
# Label enconding em Country
label_encoder = LabelEncoder()
X_train['country'] = label_encoder.fit_transform(X_train['country'])
X_train['country'].value_counts()

In [ ]:
X_train.shape

In [ ]:
# Matriz correlação
tudo = pd.concat([X_train, y_train], axis=1)
corr_matrix = tudo.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlação')
plt.show

In [ ]:
# ads_listened_per_week|offline_listening = -0.88
# ads_listened_per_week|subscription_type = -0.68
# offline_listening|subscription_type = 0.78

In [ ]:
# Converter booleanos para inteiros (como você já fez)
X_train_vif = X_train.copy()
for col in X_train_vif.columns:
    if X_train_vif[col].dtype == 'bool':
        X_train_vif[col] = X_train_vif[col].astype(int)

# Calcular VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X_train_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_vif.values, i)
                   for i in range(X_train_vif.shape[1])]

# Ordenar por VIF para melhor visualização
vif_data = vif_data.sort_values("VIF", ascending=False)
vif_data

In [ ]:
# excluir offline_listening, subscription_type
X_train = X_train.drop(['offline_listening', 'subscription_type','user_id'], axis=1)
X_train.head()

In [ ]:
# Criando subplots lado a lado
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribuição das Variáveis - X_train', fontsize=16, fontweight='bold')

# Plotando cada variável em seu próprio subplot com KDE
sns.histplot(ax=axes[0,0], data=X_train, x='age', bins=20, kde=True, color='blue', edgecolor='black')
axes[0,0].set_title('Idade\n(16-59 anos)')
axes[0,0].set_ylabel('Frequência')

sns.histplot(ax=axes[0,1], data=X_train, x='listening_time', bins=20, kde=True, color='red', edgecolor='black')
axes[0,1].set_title('Tempo de Escuta\n(10-299 min)')

sns.histplot(ax=axes[0,2], data=X_train, x='songs_played_per_day', bins=20, kde=True, color='green', edgecolor='black')
axes[0,2].set_title('Músicas por Dia\n(1-99)')

sns.histplot(ax=axes[1,0], data=X_train, x='skip_rate', bins=20, kde=True, alpha=0.7, color='orange', edgecolor='black')
axes[1,0].set_title('Taxa de Skip\n(0-0.6)')
axes[1,0].set_ylabel('Frequência')
axes[1,0].set_xlabel('Valor')

# Adicionando a nova variável ads_listened_per_week
sns.histplot(ax=axes[1,1], data=X_train, x='ads_listened_per_week', bins=20, kde=True, color='purple', edgecolor='black')
axes[1,1].set_title('Anúncios por Semana')
axes[1,1].set_xlabel('Quantidade')

# Remover o subplot vazio (agora é o [1,2])
fig.delaxes(axes[1,2])

plt.tight_layout()
plt.show()

In [ ]:
X_train['ads_listened_per_week'].describe()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Pré-processador para treino
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['age', 'listening_time', 'songs_played_per_day', 'skip_rate', 'ads_listened_per_week']),
    ],
    remainder='passthrough'
)

# Aplicar FIT_TRANSFORM apenas no treino
X_train_processed = pd.DataFrame(
    preprocessor.fit_transform(X_train),
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)
X_train = X_train_processed
X_train.head()

In [ ]:
X_train['num__ads_listened_per_week'].describe()

In [ ]:
# grafico de barras com percentual para a variavel alvo
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='is_churned', data=base_train, palette='viridis')
plt.title('Distribuição da Variável Alvo')
plt.xlabel('Churn')
plt.ylabel('Frequência')

# Adicionar percentuais nas barras
total = len(base_train)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()/2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')

plt.show()

In [ ]:
# aplicando o smote
#from imblearn.over_sampling import SMOTE
#smote = SMOTE(random_state=42)
#X_treino_balanced, y_treino_balanced = smote.fit_resample(X_train, y_train)
#base_treino_final = pd.concat([
#    pd.DataFrame(X_treino_balanced),
#    pd.DataFrame(y_treino_balanced)
#], axis=1)
#base_treino_final['is_churned'].value_counts()
#------------------------------------------------------------------------------
# tentando undersampling
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_treino_balanced, y_treino_balanced = rus.fit_resample(X_train, y_train)
base_treino_final = pd.concat([
    pd.DataFrame(X_treino_balanced),
    pd.DataFrame(y_treino_balanced)
], axis=1)
base_treino_final['is_churned'].value_counts()

In [ ]:
base_treino_final.shape

# 3 - LINHA DE BASE (logistic regression)

In [ ]:
# Pré-processamento do teste
X_test = base_test.drop('is_churned', axis=1)
y_test = base_test['is_churned']
X_test = X_test.drop(['offline_listening', 'subscription_type', 'user_id'], axis=1)
X_test = pd.get_dummies(X_test, columns=['gender', 'device_type'], drop_first=True)
X_test['country'] = label_encoder.transform(X_test['country'])  # Use transform, não fit_transform
# Aplicar apenas TRANSFORM no teste (usando o preprocessor já treinado)
X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),  # Apenas transform, não fit_transform
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)
X_test = X_test_processed
X_test.head()

In [ ]:
# criando um modelo de benchmark de LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,classification_report
# Initialize and train the Logistic Regression model
model = LogisticRegression(random_state=42)
model.fit(X_treino_balanced, y_treino_balanced)

# Make predictions on the test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("\n" + "="*50 + "\n")
print(f"Acurácia do modelo: {accuracy:.4f}")
print("\n" + "="*50)
print(classification_report(y_test,y_pred))

# 4 - PYCARET

In [ ]:
!pip install pycaret

In [ ]:
# usando pycaret
from pycaret.classification import *

In [ ]:
experimento = setup(data=base_treino_final, target='is_churned', session_id=42)

In [ ]:
modelos = compare_models(sort = 'Accuracy', fold = 10)

In [ ]:
modelo = create_model('et')

In [ ]:
print(modelo)

In [ ]:
modelo_tunado = tune_model(modelo)

In [ ]:
predict_model(modelo)

In [ ]:
base_teste = pd.concat([
    pd.DataFrame(X_test),
    pd.DataFrame(y_test)
], axis=1)
pred = predict_model(modelo, data=base_teste)

In [ ]:
plot_model(modelo, plot = 'pipeline', )

In [ ]:
plot_model(modelo, plot = 'confusion_matrix', plot_kwargs={'percent': True})

In [ ]:
plot_model(modelo, plot = 'feature')

In [ ]:
plot_model(modelo, plot = 'parameter')

# 5 - VALIDAÇÃO

In [ ]:
validacao.head()

In [ ]:
# Aplicando as transformações do treino na validação
X_validacao = validacao.drop('is_churned', axis=1)
y_validacao = validacao['is_churned']
X_validacao = X_validacao.drop(['offline_listening', 'subscription_type','user_id'], axis=1)
X_validacao = pd.get_dummies(X_validacao, columns=['gender', 'device_type'], drop_first=True)
X_validacao['country'] = label_encoder.transform(X_validacao['country'])  # Apenas transform, não fit_transform
# Aplicar apenas TRANSFORM na validação (usando o preprocessor já treinado)
X_validacao_processed = pd.DataFrame(
    preprocessor.transform(X_validacao),  # Apenas transform, não fit_transform
    columns=preprocessor.get_feature_names_out(),
    index=X_validacao.index
)
X_validacao = X_validacao_processed
X_validacao.head()

In [ ]:
validacao_transformada = pd.concat([
    pd.DataFrame(X_validacao),
    pd.DataFrame(y_validacao)
], axis=1)
pred_validacao = predict_model(modelo, data=validacao_transformada)